# 00.8 — Build a deep-learning project from scratch (capstone)

Everything from 00.5–00.7 — shell, git, pip, packaging — assembled into the thing you actually came for: **starting a new Python deep-learning project from an empty directory**, correctly, the way `neural_data_decoding` itself is built.

Every cell below really runs. By the end there is a working project in `/tmp/tinydecoder` with:

1. Version control from minute zero, with a DL-grade `.gitignore`.
2. A `pyproject.toml` manifest and `src/` package layout.
3. A tiny PyTorch model + seeded, reproducible training.
4. A pytest test suite that actually guards something.
5. Formatting/lint checks, and the habits that keep all of it healthy.

**Prerequisites:** [00.5](00.5_the_command_line_for_matlab_users.ipynb), [00.6](00.6_git_and_github_for_matlab_users.ipynb), [00.7](00.7_pip_packaging_and_project_anatomy.ipynb). Modules 01–02 deepen the Python/PyTorch you'll see here — it's fine to read this first and return.

## Section 1 — What MATLAB does

A new MATLAB project usually starts as a folder with a script in it. Structure accretes: helper functions get their own files, a `data/` folder appears, maybe a `startup.m`. Nothing enforces a shape, nothing records dependencies, and running it on a second machine is archaeology.

The Python ecosystem's answer is a small set of **conventions** — not enforced by the language, but so widely shared that tools (pip, pytest, editors, CI) all understand them. This notebook IS those conventions, in order of application.

## Section 2 — Scaffold: directory, git, `.gitignore` first

Order matters: the `.gitignore` goes in **before** anything generates files, so nothing junk can ever slip into history (00.6 §2.3 explains why this is the expensive mistake).

In [ ]:
import pathlib, subprocess, sys

PROJ = pathlib.Path("/tmp/tinydecoder")
!rm -rf {PROJ}
(PROJ / "src" / "tinydecoder").mkdir(parents=True)
(PROJ / "tests").mkdir()

(PROJ / ".gitignore").write_text('''__pycache__/
*.pyc
.venv/
.ipynb_checkpoints/
data/
results/
*.pt
*.ckpt
.DS_Store
''')

!git -C {PROJ} init -q -b main && git -C {PROJ} config user.name "Demo" && git -C {PROJ} config user.email "demo@example.com"
!git -C {PROJ} add .gitignore && git -C {PROJ} commit -q -m "Scaffold: .gitignore before anything else"
print("scaffolded:", sorted(p.name for p in PROJ.iterdir()))

## Section 3 — The manifest and the package

The `pyproject.toml` (00.7 §2.3) declares the project; the `src/` layout (00.7 §2.5) holds the code. Our project: a tiny neural decoder — an MLP that classifies synthetic "trials", a conceptual miniature of what `neural_data_decoding` does at scale.

In [ ]:
(PROJ / "pyproject.toml").write_text('''
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "tinydecoder"
version = "0.1.0"
description = "A miniature neural decoder - curriculum capstone"
requires-python = ">=3.10"
dependencies = [
    "torch>=2.0",
    "numpy>=1.24",
]

[project.optional-dependencies]
dev = ["pytest>=7.4"]

[tool.setuptools.packages.find]
where = ["src"]
''')
print((PROJ / "pyproject.toml").read_text())

In [ ]:
(PROJ / "src" / "tinydecoder" / "__init__.py").write_text('''\"\"\"tinydecoder - a miniature neural decoder.\"\"\"

from tinydecoder.model import TinyDecoder
from tinydecoder.train import train

__all__ = ["TinyDecoder", "train"]
''')

(PROJ / "src" / "tinydecoder" / "model.py").write_text('''\"\"\"The model: a two-layer MLP classifier.\"\"\"

import torch
from torch import nn


class TinyDecoder(nn.Module):
    \"\"\"Classify feature vectors into ``num_classes`` classes.\"\"\"

    def __init__(self, in_features: int, hidden: int, num_classes: int):
        super().__init__()          # mandatory! (curriculum 01.4)
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
''')

(PROJ / "src" / "tinydecoder" / "train.py").write_text('''\"\"\"Seeded synthetic-data training - small but honest.\"\"\"

import torch
from torch import nn

from tinydecoder.model import TinyDecoder


def make_data(n: int, in_features: int, num_classes: int, seed: int):
    \"\"\"Synthetic linearly-separable classification data.\"\"\"
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, in_features, generator=g)
    w = torch.randn(in_features, num_classes, generator=g)
    y = (x @ w).argmax(dim=1)
    return x, y


def train(epochs: int = 30, seed: int = 0) -> float:
    \"\"\"Train on synthetic data; return final accuracy. Fully seeded.\"\"\"
    torch.manual_seed(seed)                    # reproducibility, part 1
    x, y = make_data(512, in_features=16, num_classes=4, seed=seed)
    model = TinyDecoder(in_features=16, hidden=32, num_classes=4)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = (model(x).argmax(dim=1) == y).float().mean().item()
    return acc
''')
print("package written:")
!find {PROJ}/src -name "*.py" | sort

## Section 4 — Install editable, run it

Same command as the real project (minus this venv already having torch — pip just verifies it):

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJ)],
                   capture_output=True, text=True)
print(r.stderr.strip() or "installed OK")

import site, importlib
site.main(); importlib.invalidate_caches()      # mid-session editable install (00.7 §4.2)

from tinydecoder import train
acc = train(epochs=30, seed=0)
print(f"final training accuracy: {acc:.3f}")

## Section 5 — Tests that guard something

Two tests, each protecting a different failure mode: a **shape contract** (the model produces what downstream code expects) and a **learning smoke test** (training actually improves over chance) — plus a third that pins reproducibility. These are miniatures of this repo's 792-test suite — same tool, same layout, same idea.

In [ ]:
(PROJ / "tests" / "test_tinydecoder.py").write_text('''\"\"\"Tests for tinydecoder - shape contract + learning smoke test.\"\"\"

import torch

from tinydecoder import TinyDecoder, train


def test_output_shape():
    model = TinyDecoder(in_features=16, hidden=32, num_classes=4)
    out = model(torch.randn(8, 16))
    assert out.shape == (8, 4)


def test_training_beats_chance():
    acc = train(epochs=30, seed=0)
    assert acc > 0.5          # 4 classes -> chance is 0.25


def test_training_is_reproducible():
    assert train(epochs=5, seed=123) == train(epochs=5, seed=123)
''')

r = subprocess.run([sys.executable, "-m", "pytest", str(PROJ / "tests"), "-q"],
                   capture_output=True, text=True, cwd=PROJ)
print(r.stdout[-500:])

Read the third test again — `test_training_is_reproducible` — because it encodes the whole reproducibility discipline:

* **Seed every RNG you touch** (`torch.manual_seed`, plus `numpy` and `random` if used; explicit `torch.Generator`s are even better — no global state).
* **Record versions**: `pip freeze > requirements-lock.txt` alongside published results (00.7 §2.4).
* Note the honest caveat: bit-exact reproducibility on **GPU** additionally needs `torch.use_deterministic_algorithms(True)` and can cost speed; run-to-run *statistical* similarity is the usual working standard.

## Section 6 — Quality tooling

The real project layers four automated checks; each is one command, and all are configured in `pyproject.toml` `[tool.*]` blocks:

| Tool | Catches | This repo's gate |
|---|---|---|
| `ruff format` (or `black`) | formatting arguments — by making them moot | dev extra |
| `ruff check` | lint: unused imports, undefined names, bug-prone patterns | dev extra |
| `pyright` | type errors, before running anything (01.7) | 0 errors on `src/` |
| `pytest` | behavior regressions | 792 pass |
| `interrogate` | undocumented public functions | 100% docstrings |

Live demo — ruff inspects our new project:

In [ ]:
r = subprocess.run([sys.executable, "-m", "ruff", "check", str(PROJ / "src")],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip() or "ruff: all checks passed")

The habit that makes these stick is **pre-commit hooks** — git runs the checks automatically before every commit and refuses commits that fail. One config file (`.pre-commit-config.yaml`) + `pre-commit install`, and the machine enforces what discipline forgets. (This repo wires `nbstripout` the same way — a check that runs without being remembered.)

## Section 7 — GPU/CUDA setup (the platform-specific step)

The one install step that varies by machine — `pip install torch` gives you:

| Platform | What you get | To check |
|---|---|---|
| macOS (Apple silicon) | CPU + **MPS** GPU support, automatically | `torch.backends.mps.is_available()` |
| Linux (e.g. ACCRE) | the default wheel bundles a recent **CUDA**; for a specific CUDA version use the selector at [pytorch.org/get-started](https://pytorch.org/get-started/locally/) (it generates an `--index-url` install line) | `torch.cuda.is_available()`, `nvidia-smi` in the shell |
| Windows | CPU by default; CUDA via the same selector | same |

Two rules keep you sane: **(1)** the CUDA *driver* on the machine must be at least as new as the CUDA version your torch wheel was built for (`nvidia-smi` shows the driver's ceiling); **(2)** write device-agnostic code — the `device = "cuda" if ... else "mps" if ... else "cpu"` dance from [02.4](../02_numpy_and_pytorch_basics/02.4_pytorch_tensors_intro.ipynb) — so the same code runs on the laptop and the cluster.

## Section 8 — Commit it, and map it to the real thing

The project is real: commit it as you would daily.

In [ ]:
!git -C {PROJ} add -A && git -C {PROJ} commit -q -m "tinydecoder v0.1: model, seeded training, tests"
!git -C {PROJ} log --oneline
print("---")
!git -C {PROJ} status --short && echo "clean tree — note: no __pycache__, thanks to .gitignore"


From here, `git remote add origin git@github.com:you/tinydecoder.git && git push -u origin main` publishes it (00.6 §2.6).

**The scale-up map** — every piece of the capstone is a miniature of this repository:

| tinydecoder | neural_data_decoding |
|---|---|
| `pyproject.toml`, 2 deps + 1 extra | same file, ~10 deps + dev/docs extras + 4 `[tool.*]` configs |
| `src/tinydecoder/` (3 modules) | `src/neural_data_decoding/` (data / models / training / interop / sweeps) |
| `make_data` synthetic trials | `SyntheticTrialDataset` + `MatFileTrialDataset` over real ephys `.mat` files |
| `TinyDecoder` MLP | variational GRU composite + multi-head classifier + confidence heads |
| `train()` loop, seeded | two-stage lifecycle, curricula, checkpointing, CM-table outputs |
| 3 tests | 792 tests + MATLAB-parity gates |
| (would add) configs as constants | Hydra-composable YAML in `configs/` |
| (would add) `__main__.py` CLI | `python -m neural_data_decoding train --config-name ... --fold ...` |

Nothing in the big repo is a different *kind* of thing — just more of each, with the same skeleton you just built by hand.

In [ ]:
# Cleanup: remove the capstone package from this venv (the files stay in /tmp).
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "tinydecoder"],
                   capture_output=True, text=True)
print("uninstalled tinydecoder from the venv")

## Section 9 — Common errors

### `error: Multiple top-level packages discovered in a flat-layout`

setuptools got confused by extra directories at the repo root. The `src/` layout + `[tool.setuptools.packages.find] where = ["src"]` (exactly what we wrote) prevents this class of error entirely.

### Tests pass locally, fail elsewhere

Almost always an undeclared dependency (works because YOUR env happens to have it) or an unseeded RNG. `pip install -e .` into a **fresh** venv is the honest test of the manifest.

### `ImportError` inside your own package

Use absolute imports within the package (`from tinydecoder.model import ...`) as we did — they behave identically from tests, notebooks, and the CLI. Relative imports (`from .model import ...`) are also fine *inside* the package, but never in scripts run directly ([01.5 §5](../01_python_for_matlab_users/01.5_modules_and_imports.ipynb)).

### `torch.cuda.is_available()` is False on a GPU machine

Wrong wheel (CPU-only), or driver older than the wheel's CUDA. `nvidia-smi` first: if IT doesn't show the GPU, no Python package will. Then reinstall torch with the selector's `--index-url` line for your CUDA version.

### The project works until you `git clone` it somewhere new

Something needed wasn't committed (check `git status` for untracked files) — or something committed shouldn't have been and shadows real data. `git clone` into `/tmp` and running the tests there is a five-minute self-audit worth doing before sharing.

## Section 10 — Further reading

- [PyTorch — get started locally](https://pytorch.org/get-started/locally/) — the CUDA/platform install selector.
- [pytest — getting started](https://docs.pytest.org/en/stable/getting-started.html) — the official intro; fixtures and parametrize are the next concepts worth learning.
- [pre-commit](https://pre-commit.com/) — automate the quality gates.
- [PyTorch reproducibility notes](https://pytorch.org/docs/stable/notes/randomness.html) — the full story on determinism, including the GPU caveats.
- [Ten simple rules for structuring research software (PLOS Comp Bio)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005510) — the science-facing version of everything above.

**Module 00 is complete.** Continue to **[Module 01 — Python for MATLAB users](../01_python_for_matlab_users/01.1_syntax_basics.ipynb)**, or jump to whatever the [curriculum map](../README.md) says you need.